Load the data

In [4]:
#  Loading the data with all 200 features (for 200x200 data) - alternative loading method

from scipy.io import loadmat
import torch
import numpy as np

filepath = "Input Data\FC1.mat"
fc_mat = loadmat(filepath)
fc_mat = fc_mat['FC1']  # Extract the FC1 variable from the loaded .mat file

print(fc_mat.shape)  # Should print (200, 200, N_subjects)

fc_mat_inv = np.transpose(fc_mat, (2, 0, 1))  # Transpose to (N_subjects, 200, 200)
fc_mat_inv = fc_mat_inv[:, None, :, :]  # Add a channel dimension to get (N_subjects, 1, 200, 200)
print(fc_mat_inv.shape)  # Should print (N_subjects, 1, 200, 200)

X = torch.from_numpy(fc_mat_inv).float()  # Convert to PyTorch tensor
print("Data shape:", X.shape)  # Should print (N_subjects, 1, 200, 200)

# Expected shape is (Batch size, channels, height, width)

<>:7: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
<>:7: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
C:\Users\oddafo\AppData\Local\Temp\ipykernel_14552\2830987651.py:7: SyntaxWarning: "\F" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\F"? A raw string is also an option.
  filepath = "Input Data\FC1.mat"


(200, 200, 72)
(72, 1, 200, 200)
Data shape: torch.Size([72, 1, 200, 200])


In [5]:
def check_zero_mean_unit_range(data, atol=1e-5):
    if hasattr(data, "detach"):
        arr = data.detach().cpu().numpy()
    else:
        arr = np.asarray(data)

    mean_val = arr.mean()
    min_val = arr.min()
    max_val = arr.max()

    is_zero_mean = np.isclose(mean_val, 0.0, atol=atol)
    has_minus_one = np.isclose(min_val, -1.0, atol=atol)
    has_plus_one = np.isclose(max_val, 1.0, atol=atol)

    print(f"Doing checks for zero mean and unit range:")
    print(f"mean: {mean_val:.6f}")
    print(f"standard deviation: {arr.std():.6f}")
    print(f"min:  {min_val:.6f}")
    print(f"max:  {max_val:.6f}")
    print(f"zero mean: {is_zero_mean}")
    print(f"min == -1: {has_minus_one}")
    print(f"max ==  1: {has_plus_one}")

    return is_zero_mean and has_minus_one and has_plus_one

# Example:
# check_zero_mean_unit_range(X)

def z_normalize(data, eps=1e-8):
    if hasattr(data, "detach"):
        mean = data.mean(dim=(1, 2, 3), keepdim=True)
        std = data.std(dim=(1, 2, 3), keepdim=True, unbiased=False).clamp_min(eps)
        return (data - mean) / std

    arr = np.asarray(data)
    mean = arr.mean(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = arr.std(axis=tuple(range(1, arr.ndim)), keepdims=True)
    std = np.maximum(std, eps)
    return (arr - mean) / std


In [8]:
check_zero_mean_unit_range(X)
z_normalized_X = z_normalize(X)
check_zero_mean_unit_range(z_normalized_X)

Doing checks for zero mean and unit range:
mean: 0.008007
min:  -0.825187
max:  1.000000
zero mean: False
min == -1: False
max ==  1: True
Doing checks for zero mean and unit range:
mean: -0.000000
min:  -2.785055
max:  3.682692
zero mean: True
min == -1: False
max ==  1: False


False

In [6]:
import pandas as pd

def read_csv_and_column_min_max(csv_path):
    df = pd.read_csv(csv_path)
    min_max = pd.DataFrame({
        "min": df.min(numeric_only=True),
        "max": df.max(numeric_only=True)
    })
    return df, min_max

# Example:
# df, column_stats = read_csv_and_column_min_max("your_file.csv")
# print(column_stats)

In [3]:
csv_path = "C:\Mats og Odd Arne\Prosjektoppgave\ISC_data\Beh.csv"

df, column_stats = read_csv_and_column_min_max(csv_path)
for value in  column_stats.itertuples():
    print(f"{value.Index}: min={value.min}, max={value.max}")

Subject: min=11001.0, max=12334.0
Group: min=1.0, max=2.0
Age: min=19.0, max=82.0
Order: min=1.0, max=141.0
Sex: min=1.0, max=2.0
Relationshipstatus: min=1.0, max=5.0
Avg_Sleep: min=4.0, max=10.5
EnglishYearsSpeaking: min=0.0, max=43.0
YearsEducation: min=0.0, max=29.0
PsychDiagnosis: min=0.0, max=1.0
Medicine: min=0.0, max=1.0
NeuroImpairment: min=0.0, max=1.0
EV_base: min=1.0, max=9.0
EV_neu: min=1.0, max=9.0
EV_neg: min=1.0, max=9.0
ER_base: min=1.0, max=9.0
ER_neu: min=1.0, max=9.0
ER_neg: min=1.0, max=9.0
dEV_neu: min=-4.0, max=6.0
dEV_neg: min=-8.0, max=6.0
dER_neu: min=-3.0, max=5.0
dER_neg: min=-4.0, max=8.0
Q1_DASSDepression: min=0.0, max=18.0
Q1_DASSAnxiety: min=0.0, max=14.0
Q1_DASSStress: min=0.0, max=19.0
Q2_DERSScore: min=47.0, max=119.0
Q2_DERSNONACCEPT: min=6.0, max=23.0
Q2_DERSGOALS: min=5.0, max=25.0
Q2_DERSIMPULSE: min=6.0, max=22.0
Q2_DERSAWARENESS: min=6.0, max=28.0
Q2_DERSSTRATEGIES: min=8.0, max=31.0
Q2_DERSCLARITY: min=5.0, max=21.0
Q3_GHQ28Score: min=31.0, max=

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\oddafo\AppData\Local\Temp\ipykernel_4236\3478438818.py:1: SyntaxWarning: invalid escape sequence '\M'
  csv_path = "C:\Mats og Odd Arne\Prosjektoppgave\ISC_data\Beh.csv"


In [19]:
import scipy.io as sio
import pandas as pd
import numpy as np
import re


def extract_id_from_mat_name(text):
    """
    Extract subject ID from text between 'sub-' and '-task'.
    Example:
        'something_sub-0123-task-rest_run-1_bold.mat' -> '0123'
    """
    start_str = "sub-"
    end_str = "_task"
    
    match = re.search(r"sub-(.*?)_task", text)
    return match.group(1).strip() if match else None


def normalize_id(x, remove_leading_zeros=False):
    """
    Convert IDs to a common format so strings/numbers compare correctly.
    """
    x = str(x).strip()

    # Convert things like 123.0 -> 123
    if re.fullmatch(r"\d+\.0", x):
        x = x[:-2]

    if remove_leading_zeros and x.isdigit():
        x = str(int(x))

    return x


def flatten_subject_names(arr):
    """
    Flatten MATLAB subject_names into a list of strings.
    Works for common scipy.io.loadmat outputs.
    """
    arr = np.asarray(arr)
    values = []

    for item in arr.flat:
        if isinstance(item, np.ndarray):
            if item.size == 1:
                values.append(str(item.item()).strip())
            else:
                values.append("".join(str(x) for x in item.flat).strip())
        else:
            values.append(str(item).strip())

    return values


def load_ids_from_mat(mat_file, remove_leading_zeros=False):
    """
    Load subject IDs from subject_names in a .mat file.
    """
    mat_data = sio.loadmat(mat_file)

    if "subject_names" not in mat_data:
        raise ValueError(f"'subject_names' not found in {mat_file}")

    subject_names = flatten_subject_names(mat_data["subject_names"])

    ids = set()
    for name in subject_names:
        sid = extract_id_from_mat_name(name)
        if sid is not None:
            ids.add(normalize_id(sid, remove_leading_zeros))

    return ids


def load_ids_from_csv(csv_file, remove_leading_zeros=False):
    """
    Load IDs from a CSV file.
    This searches all cells and keeps values that look like IDs.
    If your CSV has a specific ID column, I can make this stricter.
    """
    df = pd.read_csv(csv_file)
    ids = set()

    for col in df.columns:
        for value in df[col].dropna():
            val = normalize_id(value, remove_leading_zeros)
            ids.add(val)

    return ids


def compare_two_mats_against_csv(mat_file_1, mat_file_2, csv_file, remove_leading_zeros=False):
    """
    Compare the COMBINED subject IDs from two MAT files against one CSV file.
    Returns:
        only_in_combined_mats
        only_in_csv
    """
    mat_ids_1 = load_ids_from_mat(mat_file_1, remove_leading_zeros)
    mat_ids_2 = load_ids_from_mat(mat_file_2, remove_leading_zeros)

    combined_mat_ids = mat_ids_1.union(mat_ids_2)
    csv_ids = load_ids_from_csv(csv_file, remove_leading_zeros)

    only_in_combined_mats = sorted(combined_mat_ids - csv_ids)
    only_in_csv = sorted(csv_ids - combined_mat_ids)

    print(f"\nTotal IDs in {mat_file_1}: {len(mat_ids_1)}")
    print(f"Total IDs in {mat_file_2}: {len(mat_ids_2)}")
    print(f"Total unique IDs in combined MAT files: {len(combined_mat_ids)}")
    print(f"Total IDs in CSV: {len(csv_ids)}")

    print("\nIDs in combined MAT files but not in CSV:")
    for sid in only_in_combined_mats:
        print(sid)

    print("\nIDs in CSV but not in combined MAT files:")
    for sid in only_in_csv:
        print(sid)


    return only_in_combined_mats, only_in_csv

In [21]:
Included_subjects_path = "Dummy Test Data\\Included subjects.csv"
matlab_path_young = "Input Data\\YoungAge_combined_run1_400_noZ.mat"
matlab_path_old = "Input Data\\OldAge_combined_run1_400_noZ.mat"


missing = compare_two_mats_against_csv(
    mat_file_1=matlab_path_young,
    mat_file_2=matlab_path_old,
    csv_file=Included_subjects_path
)

print("\nSummary of missing IDs:")
print(f"IDs in combined MAT files but not in CSV: {len(missing[0])}")
print(f"IDs in CSV but not in combined MAT files: {len(missing[1])}")


Total IDs in Input Data\YoungAge_combined_run1_400_noZ.mat: 72
Total IDs in Input Data\OldAge_combined_run1_400_noZ.mat: 69
Total unique IDs in combined MAT files: 141
Total IDs in CSV: 140

IDs in combined MAT files but not in CSV:
12179

IDs in CSV but not in combined MAT files:

Summary of missing IDs:
IDs in combined MAT files but not in CSV: 1
IDs in CSV but not in combined MAT files: 0


In [25]:
def check_missing_id_location(missing_id, mat_file_1, mat_file_2, csv_file,
                              id_column=None, remove_leading_zeros=False):
    """
    Check which dataset contains the given missing ID.
    """
    missing_id = normalize_id(missing_id, remove_leading_zeros)

    ids_mat_1 = load_ids_from_mat(mat_file_1, remove_leading_zeros)
    ids_mat_2 = load_ids_from_mat(mat_file_2, remove_leading_zeros)
    ids_csv = load_ids_from_csv(csv_file, remove_leading_zeros)

    in_mat_1 = missing_id in ids_mat_1
    in_mat_2 = missing_id in ids_mat_2
    in_csv = missing_id in ids_csv

    print(f"ID {missing_id}:")
    print(f"- In {mat_file_1}: {in_mat_1}")
    print(f"- In {mat_file_2}: {in_mat_2}")
    print(f"- In {csv_file}: {in_csv}")

    return {
        mat_file_1: in_mat_1,
        mat_file_2: in_mat_2,
        csv_file: in_csv
    }

def check_single_missing_id(mat_file_1, mat_file_2, csv_file,
                            id_column=None, remove_leading_zeros=False):
    ids_mat_1 = load_ids_from_mat(mat_file_1, remove_leading_zeros)
    ids_mat_2 = load_ids_from_mat(mat_file_2, remove_leading_zeros)
    ids_csv = load_ids_from_csv(csv_file, id_column, remove_leading_zeros)

    combined_mat_ids = ids_mat_1.union(ids_mat_2)
    symmetric_diff = combined_mat_ids.symmetric_difference(ids_csv)

    if len(symmetric_diff) != 1:
        raise ValueError(f"Expected exactly one missing ID, found {len(symmetric_diff)}: {sorted(symmetric_diff)}")

    missing_id = next(iter(symmetric_diff))

    return check_missing_id_location(
        missing_id,
        mat_file_1,
        mat_file_2,
        csv_file,
        id_column=id_column,
        remove_leading_zeros=remove_leading_zeros
    )

In [28]:
check_missing_id_location(
    missing_id=11113,  # Replace with the actual missing ID you want to check
    mat_file_1=matlab_path_young,
    mat_file_2=matlab_path_old,
    csv_file=Included_subjects_path
)

ID 11113:
- In Input Data\YoungAge_combined_run1_400_noZ.mat: True
- In Input Data\OldAge_combined_run1_400_noZ.mat: False
- In Dummy Test Data\Included subjects.csv: True


{'Input Data\\YoungAge_combined_run1_400_noZ.mat': True,
 'Input Data\\OldAge_combined_run1_400_noZ.mat': False,
 'Dummy Test Data\\Included subjects.csv': True}

In [36]:
import scipy.io as sio
import numpy as np
import re


def remove_one_subject_from_mat(mat_file, output_file, subject_id, data_key):
    """
    Remove one subject from a .mat dataset using the subject ID.

    Parameters
    ----------
    mat_file : str
        Input .mat file
    output_file : str
        Output .mat file
    subject_id : str or int
        Subject ID to remove
    data_key : str
        Key for the FC data, e.g. 'run1_data' or 'run2_data'
    """

    subject_id = str(subject_id).strip()
    mat = sio.loadmat(mat_file)

    subject_names_raw = mat["subject_names"].flatten()
    subject_names = []

    for x in subject_names_raw:
        if isinstance(x, np.ndarray):
            subject_names.append(str(x.item()))
        else:
            subject_names.append(str(x))

    subject_ids = []
    for name in subject_names:
        match = re.search(r"sub-(.*?)_task", name)
        subject_ids.append(match.group(1) if match else None)

    if subject_id not in subject_ids:
        raise ValueError(f"Subject ID {subject_id} not found in {mat_file}")

    remove_idx = subject_ids.index(subject_id)
    keep_idx = [i for i in range(len(subject_ids)) if i != remove_idx]

    new_subject_names = np.array(subject_names, dtype=object)[keep_idx]
    new_data = mat[data_key][keep_idx]

    sio.savemat(output_file, {
        "subject_names": new_subject_names,
        data_key: new_data
    })

    print(f"Removed subject {subject_id} from {mat_file}")
    print(f"Saved cleaned file to {output_file}")

import pandas as pd

def remove_one_subject_from_csv(csv_file, output_file, subject_id, id_column):
    subject_id = str(subject_id).strip()
    df = pd.read_csv(csv_file)
    df = df[df[id_column].astype(str).str.strip() != subject_id]
    df.to_csv(output_file, index=False)
    print(f"Removed subject {subject_id} from {csv_file}")
    print(f"Saved cleaned file to {output_file}")

In [34]:
missing_id = "11113"

remove_one_subject_from_mat(
    "Input Data\\YoungAge_combined_run1_400_noZ.mat",
    "Input Data\\YoungAge_combined_run1_400_noZ_clean.mat",
    missing_id,
    data_key="run1_data"
)

remove_one_subject_from_mat(
    "Input Data\\YoungAge_combined_run2_400_noZ.mat",
    "Input Data\\YoungAge_combined_run2_400_noZ_clean.mat",
    missing_id,
    data_key="run2_data"
)

Removed subject 11113 from Input Data\YoungAge_combined_run1_400_noZ.mat
Saved cleaned file to Input Data\YoungAge_combined_run1_400_noZ_clean.mat
Removed subject 11113 from Input Data\YoungAge_combined_run2_400_noZ.mat
Saved cleaned file to Input Data\YoungAge_combined_run2_400_noZ_clean.mat


In [39]:
remove_one_subject_from_csv(
    "Dummy Test Data\\Dummy_Beh.csv",
    "Dummy Test Data\\Dummy_Beh_clean.csv",
    missing_id,
    id_column="Subject"
)


Removed subject 11113 from Dummy Test Data\Dummy_Beh.csv
Saved cleaned file to Dummy Test Data\Dummy_Beh_clean.csv
